# Clasificación de reclamos CMF con Jev

**Objetivo:** clasificar los reclamos ciudadanos presentados a la CMF (Comisión para el Mercado Financiero) según el mercado que corresponde. Las categorías provienen del dataset ya etiquetado por analistas.

**Requisito:** un archivo `.env` en esta carpeta, con la clave `JEV_API_KEY`. El notebook la lee al iniciar.

## Configuración

In [1]:
import os
from pathlib import Path

import pandas as pd
from typesafe_sdk import Choice, TypeSafeClient


def ruta_env() -> Path:
    inicios = [Path.cwd().resolve()]
    nb = globals().get("__vsc_ipynb_file__")
    if isinstance(nb, str) and nb:
        inicios.insert(0, Path(nb).resolve().parent)
    vistos = set()
    for inicio in inicios:
        for carpeta in [inicio, *inicio.parents]:
            if carpeta in vistos:
                continue
            vistos.add(carpeta)
            candidato = carpeta / ".env"
            if (carpeta / "clasificar_reclamos_jev.ipynb").is_file() and candidato.is_file():
                return candidato
    raise FileNotFoundError("No está .env en la carpeta del notebook.")


for linea in ruta_env().read_text(encoding="utf-8").splitlines():
    linea = linea.strip()
    if not linea or linea.startswith("#") or "=" not in linea:
        continue
    linea = linea.removeprefix("export ")
    clave, valor = linea.split("=", 1)
    valor = valor.strip()
    if len(valor) >= 2 and valor[0] == valor[-1] and valor[0] in {"'", '"'}:
        valor = valor[1:-1]
    os.environ[clave.strip()] = valor

api_key = os.environ.get("JEV_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("Defina JEV_API_KEY en .env.")
print("JEV_API_KEY leída desde .env.")
client = TypeSafeClient(api_key=api_key)

JEV_API_KEY leída desde .env.


## Carga de datos

Las categorías se obtienen de la columna `MERCADO_ANALISTA`. Se eliminan espacios sobrantes.

In [2]:
con_clas = pd.read_excel("reclamos_20201221_con_clas.xlsx")
sin_clas = pd.read_excel("reclamos_20201221_sin_clas.xlsx")
con_clas["MERCADO_ANALISTA"] = con_clas["MERCADO_ANALISTA"].str.strip()

CATEGORIAS = sorted(con_clas["MERCADO_ANALISTA"].unique())
con_clas["MERCADO_ANALISTA"].value_counts()

MERCADO_ANALISTA
APIA -Reclamo Seguros    662
Reclamo Valores          337
Name: count, dtype: int64

## Función de clasificación

Jev recibe la descripción y la petición de cada reclamo. Devuelve una categoría de la lista y su confianza, entre 0 y 1.

In [3]:
PREGUNTA = {
    "mercado": Choice(
        instructions="¿A qué mercado supervisado por la CMF corresponde este reclamo ciudadano?",
        criteria={c: None for c in CATEGORIAS},
    )
}


def clasificar(fila: pd.Series) -> pd.Series:
    estado = {
        "descripcion": fila["DESCRIPCION_CIUDADANO"],
        "peticion": fila["PETICION_CIUDADANO"],
    }
    r = client.system_one(state=estado, questions=PREGUNTA).choices["mercado"]
    return pd.Series({"MERCADO_JEV": r.choice, "CONFIANZA_JEV": r.confidence})

## Validación

Se clasifica una muestra de reclamos ya etiquetados. La exactitud y la matriz de confusión miden la calidad antes de clasificar los reclamos nuevos.

In [4]:
N_VALIDACION = 30

muestra = con_clas.sample(N_VALIDACION, random_state=42)
validacion = muestra.join(muestra.apply(clasificar, axis=1))

exactitud = (validacion["MERCADO_JEV"] == validacion["MERCADO_ANALISTA"]).mean()
print(f"Exactitud: {exactitud:.1%}")
pd.crosstab(validacion["MERCADO_ANALISTA"], validacion["MERCADO_JEV"])

Exactitud: 100.0%


MERCADO_JEV,APIA -Reclamo Seguros,Reclamo Valores
MERCADO_ANALISTA,,
APIA -Reclamo Seguros,17,0
Reclamo Valores,0,13


## Clasificación de reclamos sin categoría

In [5]:
resultado = sin_clas.join(sin_clas.apply(clasificar, axis=1))
resultado["MERCADO_JEV"].value_counts()

MERCADO_JEV
APIA -Reclamo Seguros    70
Reclamo Valores          30
Name: count, dtype: int64

Los 10 casos con menor confianza requieren revisión manual.

In [6]:
resultado.nsmallest(10, "CONFIANZA_JEV")[
    ["CASO_ID", "MERCADO_JEV", "CONFIANZA_JEV", "DESCRIPCION_CIUDADANO"]
]

,CASO_ID,MERCADO_JEV,CONFIANZA_JEV,DESCRIPCION_CIUDADANO
73,731652,APIA -Reclamo Seguros,0.18,"La administradora Andes, antes detallada, ha h..."
28,361042,Reclamo Valores,0.28,Hace unos meses abrí una cuenta corriente ya q...
66,565303,Reclamo Valores,0.28,"El día 7 de enero solicité un giro, en oficina..."
61,418710,Reclamo Valores,0.32,"Nuestra empresa, XX opera desde hace bastante ..."
76,315391,APIA -Reclamo Seguros,0.42,Mi reclamo del 28/05/2012 y OFOR XX de la SVS...
56,280557,Reclamo Valores,0.44,Abrí una cuenta de Ahorro en el Banco Edwards ...
18,364529,Reclamo Valores,0.66,TENGO MI APV (AHORRO PREVISIONAL VOLUNTARIO) E...
63,499292,Reclamo Valores,0.70,"En enero del 2015, presenté reclamo N°XX y aún..."
82,566355,Reclamo Valores,0.70,"FACTORIZAMOS CON TANNER A TRANSELEC, Tanner no..."
87,512099,APIA -Reclamo Seguros,0.76,el 28 de abril me llamo banco estado cobranzas...


## Exportación

In [7]:
resultado.to_excel("reclamos_20201221_clasificados_jev.xlsx", index=False)
client.close()